# AELIONIX BLACKFORGE — Phase 5 Colab Validation

This notebook performs a deterministic, one-click validation of the Blackforge **Reconnaissance Capability Foundation** (Phase 5).

**What this validates:**
- Repository integrity and commit verification
- Dependency installation (runtime + dev extras)
- All Blackforge imports, including the new `blackforge.recon` modules
- Full automated test suite (recon included)
- Bootstrap + health verification, including the new `recon_ready` check
- Six typed reconnaissance capabilities (host discovery, service enumeration, technology identification, DNS, HTTP metadata, TLS metadata)
- The full recon pipeline: capability -> mock tool -> normalization -> evidence (artifact + typed observations, `DERIVED_FROM`) -> world model -> best-effort memory link
- IDEMPOTENT reruns: identical evidence rows, unchanged world model state (dedup, no invented data)
- **No generic execution surface**: only the six typed capability contracts exist; unknown capabilities are rejected
- Authorization enforced *before* any tool run (out-of-scope targets and out-of-scope capabilities are denied)
- Mission isolation and restart persistence across fresh SQLite connections
- Confidence policy: ACTIVE direct observations are HIGH confidence; passive/normalized data is LOW; technology identification is MEDIUM



---

In [ ]:
import sys
import platform

print("Blackforge Phase 5 Colab Validation (Reconnaissance Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")


---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

REPO_DIR = Path("/content/blackforge").resolve()
if not (REPO_DIR / "pyproject.toml").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/<ORG>/blackforge.git",
         str(REPO_DIR)],
        check=True,
    )
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)


---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)


---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet


---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.memory",
    "blackforge.evidence",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.canonical",
    "blackforge.world_model.rules",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.world_model.materializer",
    "blackforge.mission.manager",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.validator",
    "blackforge.recon",
    "blackforge.recon.models",
    "blackforge.recon.mock",
    "blackforge.recon.normalization",
    "blackforge.recon.evidence",
    "blackforge.recon.materializer",
    "blackforge.recon.capabilities",
    "blackforge.recon.engine",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Recon module imports: PASS")


---

In [ ]:
!pytest -q -x --disable-warnings --timeout=600


---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase5_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("evidence_store_ready", "evidence_memory_link_ready", "memory_ready",
            "world_model_ready", "recon_ready"):
    assert verification[key], f"{key} must be True"

BOOTSTRAP_OK = app.healthy() and verification["recon_ready"]

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (recon ready): PASS")


---

In [ ]:
from blackforge.recon.models import ReconMode, ReconRequest, ReconStatus
from blackforge.scope.models import TargetScope, Target, detect_target_type

def _target(value: str) -> Target:
    return Target(value=value, target_type=detect_target_type(value))

MID = "mission_phase5_recon"
scope = TargetScope(
    mission_id=MID,
    allowed_targets=[_target(t) for t in ("web.example.com", "mail.example.com", "db.example.com")],
)
req = ReconRequest(mission_id=MID, scope=scope, max_observations=2000)

engine = app.recon_engine
assert engine is not None and len(engine.capabilities) == 6
expected = sorted([
    "recon.host_discovery",
    "recon.service_discovery",
    "recon.technology_identification",
    "recon.dns",
    "recon.http_metadata",
    "recon.tls_metadata",
])
ids_seen = sorted(c.capability_id for c in engine.capabilities)
assert ids_seen == expected, (ids_seen, expected)
print("Registered reconnaissance capabilities:", ", ".join(c.capability_id for c in engine.capabilities))

_meta_by_id = {c.capability_id: c.meta() for c in engine.capabilities}
for capability_id in expected:
    meta = _meta_by_id[capability_id]
    print(
        f"  {meta.id:<34} risk={meta.risk_level.value:<6} mode={meta.mode.value:<7} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )
CAPS_OK = True

res = engine.discover_hosts(req, "web.example.com")
assert res.status == ReconStatus.SUCCESS
assert res.capability_id == "recon.host_discovery"
print(
    f"\nhost_discovery web.example.com -> status={res.status.value}, "
    f"observations={len(res.observations)}, evidence_rows={len(res.evidence_ids)}"
)
print("Host:", res.observations[0].host, "ips=", res.observations[0].ip_addresses)
assert res.observations[0].ip_addresses[0].startswith("192.0.2.")


---

In [ ]:
from blackforge.evidence.models import EvidenceRelation
from blackforge.world_model.query import RelationshipQuery

res_dns = engine.inspect_dns(req, "web.example.com")
res_svc = engine.enumerate_services(req, "web.example.com")
res_tec = engine.identify_technologies(req, "web.example.com")
res_http = engine.inspect_http_metadata(req, "web.example.com")
res_tls = engine.inspect_tls(req, "web.example.com")

runs = (res_dns, res_svc, res_tec, res_http, res_tls)
kinds = sorted({o.kind for r in runs for o in r.observations})
print("Pipeline observation kinds observed:", ", ".join(kinds))
for expected_kind in ("dns", "service", "technology", "http"):
    assert expected_kind in kinds, expected_kind
assert res_tls.observations and res_tls.observations[0].tls_version
print("TLS observation:", res_tls.observations[0].tls_version, "on endpoint",
      res_tls.observations[0].host, ":" + str(res_tls.observations[0].port))

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
count_artifacts = 0
for r in runs:
    artifact = r.evidence_ids[0]
    count_artifacts += 1
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 1
print(f"DERIVED_FROM links: {count_obs} observation rows -> {count_artifacts} artifacts: PASS")

# World model materialized from observations.
wm = app.world_model
entity_count_a = wm.count_entities(MID)
rels_a = len(wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
print(f"World model after pipeline: entities={entity_count_a}, relationships={rels_a}")
assert entity_count_a >= 4 and rels_a >= 1

# Rerun -> perfectly idempotent: identical evidence rows, no new world records.
rerun = engine.inspect_dns(req, "web.example.com")
assert [str(x) for x in rerun.evidence_ids] == [str(x) for x in res_dns.evidence_ids]
assert wm.count_entities(MID) == entity_count_a
assert len(wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))) == rels_a
DEDUP_OK = True
print("Rerun dedup (identical evidence ids, unchanged world state): PASS")


---

In [ ]:
from blackforge.core.errors import AuthorizationError

# 1) Target outside the scope is denied BEFORE any tool runs.
denied_out = False
try:
    engine.discover_hosts(req, "scanme.example.org")
except AuthorizationError:
    denied_out = True
assert denied_out
print("Out-of-scope target denied before tool execution: PASS")

# 2) Capability outside allowed_capabilities is denied.
cap_scope = TargetScope(
    mission_id=MID,
    allowed_targets=[_target("web.example.com")],
    allowed_capabilities=["recon.dns"],
)
cap_req = ReconRequest(mission_id=MID, scope=cap_scope)
denied_cap = False
try:
    engine.enumerate_services(cap_req, "web.example.com")
except AuthorizationError:
    denied_cap = True
assert denied_cap
print("Capability outside scope denied: PASS")

# 3) Mission isolation: work under a second mission is fully disjoint.
MID2 = "mission_phase5_recon_other"
scope2 = TargetScope(mission_id=MID2, allowed_targets=[_target("web.example.com")])
req2 = ReconRequest(mission_id=MID2, scope=scope2)
res2 = engine.inspect_dns(req2, "web.example.com")
other_ids = {str(x) for x in res2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in res_dns.evidence_ids})
assert app.evidence_store.count(MID2) == len(res2.evidence_ids)
assert wm.count_entities(MID2) >= 1
print("Mission isolation: second mission produced its own evidence/world rows: PASS")
MID_ISOLATION_OK = True


---

In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore
from blackforge.world_model.models import EntityType
from blackforge.world_model.query import RelationshipQuery

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == wm.count_entities(MID)
host_entity = fresh_wm.find_entity(MID, EntityType.ASSET, "web.example.com")
persisted_host = (
    host_entity is not None
    and "192.0.2.10" in host_entity.properties.get("ip_addresses", [])
)
assert persisted_ev and persisted_wm and persisted_host
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm and persisted_host

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")

for _p in sorted(DBROOT.glob("*.db")):
    _p.unlink(missing_ok=True)
print("Disposable database files removed.")


---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "recon" / "engine.py").exists(),
    "phase5_modules": bool(
        (REPO_DIR / "blackforge" / "recon" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "recon" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "recon" / "materializer.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_recon_ready": BOOTSTRAP_OK,
    "no_generic_executor": CAPS_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "world_materialized": entity_count_a >= 4,
    "dedup_idempotent": DEDUP_OK,
    "scope_authorization": denied_out,
    "capability_authorization": denied_cap,
    "mission_isolation": MID_ISOLATION_OK,
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_recon_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = phase_checks["scope_authorization"] and phase_checks["capability_authorization"]

rows = [
    "Repository",
    "Python",
    "Hardware",
    "Installation",
    "Imports",
    "Automated tests",
    "Bootstrap",
    "Phase-specific tests",
    "Security checks",
]
failed = sum(1 for k in rows if not results[k])

print("=" * 60)
print("BLACKFORGE PHASE 5 VALIDATION")
print("=" * 60)
for k in rows:
    symbol = "PASS" if results[k] else "FAIL"
    print(f"{k:<26} {symbol}")
print("=" * 60)
print(f"OVERALL RESULT: {'PASS' if failed == 0 else 'FAIL'}")
print("=" * 60)

print()
print("Phase-specific checks:")
for key, value in phase_checks.items():
    symbol = "PASS" if value else "FAIL"
    print(f"  [{symbol}] {key}")

if failed:
    raise RuntimeError(f"Phase 5 validation failed: {failed} check(s)")
print()
print("All checks passed. Disposable database files were removed during the run.")
